# Week 01: Walking Skeleton — ScraperFlow V1

**Goal:** Build the absolute minimal end-to-end scraper. Every architectural seam (fetch/parse/store) exists as a real function boundary from day one.

**Concepts:** modules & packages, `logging`, `argparse`, `requests`, `BeautifulSoup`, `json` serialization, `pytest` basics, separation of concerns, walking skeleton pattern

---
## 1. The Walking Skeleton Pattern

A walking skeleton is a tiny implementation of a system that performs a small end-to-end function. It connects the main architectural components — even though each piece is trivial.

**Why it matters:**
- Converts guesses about how a system should work into observed facts, cheaply
- Later versions attach to existing boundaries instead of carving seams out of a monolith
- Forces you to think about wiring and integration from day one
- Reveals misunderstandings about data flow early

**ScraperFlow V1 skeleton:**
```
CLI (argparse) → fetch_page() → parse_article() → save_records() → output.json
```

Each arrow is a real function call boundary. Each function lives in its own module.

---
## 2. Python `logging`

The `logging` module provides a flexible framework for emitting log messages from Python programs. It's the standard way to instrument production code — `print()` is for throwaway scripts.

### Key Concepts
- **Loggers** — named channels that accept log records
- **Handlers** — destinations (console, file, network)
- **Formatters** — control the layout of log records
- **Levels** — DEBUG < INFO < WARNING < ERROR < CRITICAL

In [2]:
import logging
import sys

# Create a named logger (don't use the root logger)
logger = logging.getLogger("scraper")
logger.setLevel(logging.DEBUG)  # Logger level gates what gets passed to handlers

# StreamHandler — outputs to console
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)  # Handler level filters further

# Formatter — controls what each line looks like
formatter = logging.Formatter(
    "%(asctime)s | %(name)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
console_handler.setFormatter(formatter)

# Attach handler to logger
logger.addHandler(console_handler)

# Test all levels — only INFO and above will appear
logger.debug("This won't appear (below handler level)")
logger.info("Starting scrape job")
logger.warning("URL returned 404, skipping")
logger.error("Connection timeout for example.com")
logger.critical("Unrecoverable failure")

2026-07-15 12:07:02 | scraper | INFO     | Starting scrape job
2026-07-15 12:07:02 | scraper | INFO     | Starting scrape job
2026-07-15 12:07:02 | scraper | WARNING  | URL returned 404, skipping
2026-07-15 12:07:02 | scraper | WARNING  | URL returned 404, skipping
2026-07-15 12:07:02 | scraper | ERROR    | Connection timeout for example.com
2026-07-15 12:07:02 | scraper | ERROR    | Connection timeout for example.com
2026-07-15 12:07:02 | scraper | CRITICAL | Unrecoverable failure
2026-07-15 12:07:02 | scraper | CRITICAL | Unrecoverable failure


In [5]:
# Adding a file handler with a different format
file_handler = logging.FileHandler("scraper.log", mode="a")
file_handler.setLevel(logging.DEBUG)  # File gets everything

file_formatter = logging.Formatter(
    "%(asctime)s | %(name)s | %(levelname)-8s | %(filename)s:%(lineno)d | %(message)s"
)
file_handler.setFormatter(file_formatter)
logger.addHandler(file_handler)

# Now DEBUG messages go to file but not console
logger.debug("Detailed debug info — only in file")
logger.info("This goes to both console and file")

2026-07-15 12:36:19 | scraper | INFO     | This goes to both console and file
2026-07-15 12:36:19 | scraper | INFO     | This goes to both console and file


### Logging Best Practices

| Do | Don't |
|---|---|
| Use named loggers (`getLogger(__name__)`) | Use the root logger directly |
| Use lazy formatting: `logger.info("Got %d items", count)` | Use f-strings in log calls (evaluated even if filtered) |
| Set levels on handlers, not just the logger | Call `logging.basicConfig()` in library code |
| Log at appropriate levels | Log everything at INFO or DEBUG |

**Level guidelines:**
- `DEBUG` — diagnostic detail, off in production
- `INFO` — confirmation things work as expected ("started", "completed", "wrote N records")
- `WARNING` — something unexpected but recoverable ("URL skipped")
- `ERROR` — a specific operation failed ("could not fetch URL")
- `CRITICAL` — program cannot continue

---
## 3. `argparse` — CLI Argument Parsing

`argparse` separates *data* (what URLs to scrape, where to write output) from *code* (how to scrape). The CLI is the boundary between the user and the program logic.

In [6]:
import argparse

def build_parser() -> argparse.ArgumentParser:
    """Construct the argument parser for ScraperFlow CLI."""
    parser = argparse.ArgumentParser(
        description="ScraperFlow V1 — fetch, parse, and store web articles"
    )
    parser.add_argument(
        "urls",
        nargs="+",
        help="One or more URLs to scrape"
    )
    parser.add_argument(
        "--output", "-o",
        default="output.json",
        help="Output file path (default: output.json)"
    )
    parser.add_argument(
        "--verbose", "-v",
        action="store_true",
        help="Enable verbose (DEBUG) logging"
    )
    return parser

# Simulate parsing CLI args (in real usage, parse_args() reads sys.argv)
parser = build_parser()
args = parser.parse_args(["http://example.com", "http://test.com", "--output", "data.json", "--verbose"])

print(f"URLs: {args.urls}")
print(f"Output: {args.output}")
print(f"Verbose: {args.verbose}")

URLs: ['http://example.com', 'http://test.com']
Output: data.json
Verbose: True


### argparse Patterns

- **Positional args** — required, order matters: `parser.add_argument("urls", nargs="+")`
- **Optional args** — prefixed with `--`: `parser.add_argument("--output", default="out.json")`
- **Flags** — boolean switches: `parser.add_argument("--verbose", action="store_true")`
- **`nargs`** — `"+"` = one or more, `"*"` = zero or more, `"?"` = zero or one
- **`type`** — auto-conversion: `parser.add_argument("--port", type=int, default=8080)`

---
## 4. `requests` & HTTP Fundamentals

`requests` is the standard HTTP client for Python. For scraping, you mostly need `GET` requests and status code handling.

In [ ]:
import requests

def fetch_page(url: str) -> str:
    """Fetch a URL and return raw HTML. Raises on non-2xx status."""
    response = requests.get(url, timeout=10)
    response.raise_for_status()  # Raises HTTPError for 4xx/5xx
    return response.text

# Demonstration
html = fetch_page("http://example.com")
print(f"Fetched {len(html)} characters")
print(html[:200])

### HTTP Status Codes

| Range | Meaning | Example |
|---|---|---|
| 2xx | Success | 200 OK, 201 Created |
| 3xx | Redirect | 301 Moved Permanently, 304 Not Modified |
| 4xx | Client Error | 400 Bad Request, 403 Forbidden, 404 Not Found |
| 5xx | Server Error | 500 Internal Server Error, 503 Service Unavailable |

### Key `requests` features for scraping

- `timeout` — always set a timeout; otherwise the request can hang forever
- `response.raise_for_status()` — raises `HTTPError` for non-2xx responses
- `response.text` — decoded content (uses detected encoding)
- `response.content` — raw bytes
- Custom headers: `requests.get(url, headers={"User-Agent": "ScraperFlow/1.0"})`

In [7]:
# Status code classification (pure function — no network needed)
def classify_status(status_code: int) -> str:
    """Classify HTTP status code into category."""
    if 200 <= status_code < 300:
        return "success"
    elif 300 <= status_code < 400:
        return "redirect"
    elif 400 <= status_code < 500:
        return "client_error"
    elif 500 <= status_code < 600:
        return "server_error"
    else:
        return "unknown"

# Verify
assert classify_status(200) == "success"
assert classify_status(301) == "redirect"
assert classify_status(404) == "client_error"
assert classify_status(500) == "server_error"
print("All status code classifications correct")

All status code classifications correct


---
## 5. BeautifulSoup & CSS Selectors

BeautifulSoup parses HTML into a navigable tree. CSS selectors let you target specific elements.

In [8]:
from bs4 import BeautifulSoup

HTML_SAMPLE = """
<html>
<head><title>Test Article</title></head>
<body>
  <article>
    <h1 class="title">Breaking News</h1>
    <p class="author">Jane Doe</p>
    <div class="content">
      <p>First paragraph of the article.</p>
      <p>Second paragraph of the article.</p>
    </div>
    <span class="date">2024-01-15</span>
  </article>
</body>
</html>
"""

soup = BeautifulSoup(HTML_SAMPLE, "html.parser")

# select_one — returns first match or None
title = soup.select_one("h1.title")
print(f"Title: {title.get_text()}")

# select — returns list of all matches
paragraphs = soup.select("div.content p")
print(f"Paragraphs: {[p.get_text() for p in paragraphs]}")

# Attribute access
author = soup.select_one(".author")
print(f"Author: {author.get_text()}")

Title: Breaking News
Paragraphs: ['First paragraph of the article.', 'Second paragraph of the article.']
Author: Jane Doe


In [9]:
def parse_article(html: str) -> dict:
    """Extract structured data from an article HTML page."""
    soup = BeautifulSoup(html, "html.parser")
    
    title_el = soup.select_one("h1.title")
    author_el = soup.select_one(".author")
    content_els = soup.select("div.content p")
    date_el = soup.select_one(".date")
    
    return {
        "title": title_el.get_text() if title_el else None,
        "author": author_el.get_text() if author_el else None,
        "content": "\n".join(p.get_text() for p in content_els) if content_els else None,
        "date": date_el.get_text() if date_el else None,
    }

result = parse_article(HTML_SAMPLE)
print(result)

{'title': 'Breaking News', 'author': 'Jane Doe', 'content': 'First paragraph of the article.\nSecond paragraph of the article.', 'date': '2024-01-15'}


### CSS Selector Reference

| Selector | Meaning | Example |
|---|---|---|
| `tag` | Element by tag name | `h1`, `p`, `div` |
| `.class` | Element by class | `.title`, `.content` |
| `#id` | Element by id | `#main-article` |
| `parent child` | Descendant | `div.content p` |
| `parent > child` | Direct child only | `article > h1` |
| `[attr]` | Has attribute | `[href]`, `[data-id]` |
| `[attr=val]` | Attribute equals | `[class="title"]` |

### Safe extraction pattern

Always check for `None` before calling `.get_text()` — a missing element means the page structure differs from expectation, not that the code is broken.

In [10]:
def safe_extract(html: str, selector: str) -> str | None:
    """Safely extract text from first element matching selector, or None."""
    soup = BeautifulSoup(html, "html.parser")
    element = soup.select_one(selector)
    return element.get_text(strip=True) if element else None

# Works when element exists
assert safe_extract(HTML_SAMPLE, "h1.title") == "Breaking News"

# Returns None when element is missing (no exception)
assert safe_extract(HTML_SAMPLE, "h2.subtitle") is None

print("safe_extract works correctly")

safe_extract works correctly


---
## 6. JSON Serialization

The `json` module handles serialization (Python → JSON string) and deserialization (JSON string → Python). It's the output format for ScraperFlow V1.

In [11]:
import json
from pathlib import Path

records = [
    {"url": "http://example.com/1", "title": "First Article", "content": "Some text"},
    {"url": "http://example.com/2", "title": "Second Article", "content": "More text"},
]

# Serialize to string
json_str = json.dumps(records, indent=2, ensure_ascii=False)
print(json_str)

[
  {
    "url": "http://example.com/1",
    "title": "First Article",
    "content": "Some text"
  },
  {
    "url": "http://example.com/2",
    "title": "Second Article",
    "content": "More text"
  }
]


In [12]:
def save_records(records: list[dict], output_path: str) -> None:
    """Write records to a JSON file."""
    path = Path(output_path)
    path.write_text(
        json.dumps(records, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

def load_records(input_path: str) -> list[dict]:
    """Read records from a JSON file."""
    path = Path(input_path)
    return json.loads(path.read_text(encoding="utf-8"))

# Round-trip test
save_records(records, "/tmp/test_output.json")
loaded = load_records("/tmp/test_output.json")
assert loaded == records
print(f"Round-trip successful: {len(loaded)} records preserved")

Round-trip successful: 2 records preserved


### JSON gotchas

- `json.dumps()` only handles basic Python types (dict, list, str, int, float, bool, None)
- `datetime`, `set`, `bytes` etc. require custom handling (a `default` function or pre-conversion)
- `ensure_ascii=False` preserves Unicode characters as-is (better for international content)
- `indent=2` produces human-readable output (larger files but easier to debug)

---
## 7. `pytest` Basics

pytest discovers and runs tests automatically. Key concepts for V1:

### Test Discovery
- Files named `test_*.py` or `*_test.py`
- Functions named `test_*`
- Classes named `Test*` (no `__init__`)

### Assertions
pytest uses plain `assert` statements — no special assertion methods needed.

### Fixtures
Fixtures provide reusable test setup via dependency injection.

In [ ]:
# Example test functions (normally in tests/test_parser.py)

def test_parse_article_extracts_title():
    """Parser extracts title from well-formed HTML."""
    html = '<html><body><article><h1 class="title">Test Title</h1></article></body></html>'
    result = parse_article(html)
    assert result["title"] == "Test Title"

def test_parse_article_handles_missing_element():
    """Parser returns None for missing elements, doesn't crash."""
    html = "<html><body><article></article></body></html>"
    result = parse_article(html)
    assert result["title"] is None
    assert result["author"] is None

# Run them inline
test_parse_article_extracts_title()
test_parse_article_handles_missing_element()
print("All tests pass")

In [ ]:
# Fixtures example (how they'd look in a test file)
# In pytest, fixtures are decorated functions that provide test data or setup

import pytest

# @pytest.fixture
# def sample_html():
#     return """
#     <html><body><article>
#         <h1 class="title">Fixture Title</h1>
#         <p class="author">Test Author</p>
#     </article></body></html>
#     """
#
# def test_with_fixture(sample_html):
#     """pytest injects the fixture by matching parameter name."""
#     result = parse_article(sample_html)
#     assert result["title"] == "Fixture Title"
#     assert result["author"] == "Test Author"

# Key fixture features:
# - Defined with @pytest.fixture decorator
# - Injected by name into test functions
# - Can use yield for setup/teardown
# - scope= controls lifetime (function, class, module, session)

print("See tests/test_parser.py for real fixture usage")

### Running pytest

```bash
# Run all tests
pytest

# Run with verbose output
pytest -v

# Run a specific file
pytest tests/test_parser.py

# Run a specific test
pytest tests/test_parser.py::test_parse_article_extracts_title

# Stop on first failure
pytest -x
```

---
## 8. Separation of Concerns & Module Organization

The V1 module layout splits code by *responsibility*, not by size:

```
scraperflow/
├── __init__.py      # Package marker
├── __main__.py      # Enables `python -m scraperflow`
├── cli.py           # Argument parsing + orchestration
├── fetcher.py       # Network I/O (requests)
├── parser.py        # HTML → structured data (BeautifulSoup)
└── storage.py       # Structured data → file (json)
```

**Why separate modules when each is small?**
- Each module has exactly one reason to change
- Testing is simpler (mock one boundary, test another)
- New developers can locate code by responsibility
- Later changes attach to an existing boundary instead of splitting a file

**The orchestration pattern:**
```python
# cli.py — wires the pieces together
def run(urls: list[str], output_path: str) -> None:
    records = []
    for url in urls:
        try:
            html = fetch_page(url)
            record = parse_article(html)
            record["url"] = url
            records.append(record)
        except Exception as e:
            logger.warning("Skipping %s: %s", url, e)
    save_records(records, output_path)
```

The CLI handles wiring and failure policy. Individual modules don't know about each other.

---
## 9. Putting It All Together — The V1 Flow

```python
# __main__.py
from scraperflow.cli import main
main()
```

Usage:
```bash
python -m scraperflow http://example.com http://test.com --output results.json
```

**Failure policy (V1):** Log and skip. A single failed URL does not crash the entire run. This is a *deliberate decision*, not an accident — document it as such.

In [ ]:
# Complete V1 flow in one cell (what the mini-project implements across modules)

import logging
import json
from pathlib import Path

import requests
from bs4 import BeautifulSoup

logger = logging.getLogger("scraperflow")

def fetch_page(url: str) -> str:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.text

def parse_article(html: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.select_one("title")
    body_el = soup.select_one("body")
    return {
        "title": title_el.get_text(strip=True) if title_el else None,
        "content": body_el.get_text(strip=True)[:500] if body_el else None,
    }

def save_records(records: list[dict], output_path: str) -> None:
    Path(output_path).write_text(
        json.dumps(records, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

def run(urls: list[str], output_path: str) -> None:
    records = []
    for url in urls:
        try:
            logger.info("Fetching %s", url)
            html = fetch_page(url)
            record = parse_article(html)
            record["url"] = url
            records.append(record)
            logger.info("Parsed: %s", record["title"])
        except Exception as e:
            logger.warning("Skipping %s: %s", url, e)
    save_records(records, output_path)
    logger.info("Wrote %d records to %s", len(records), output_path)

# Demo run
run(["http://example.com"], "/tmp/scraperflow_demo.json")
print(Path("/tmp/scraperflow_demo.json").read_text())

---
## Key Takeaways

1. **Walking skeleton first** — get the full pipeline working end-to-end before optimizing any piece
2. **Logging over print** — `logging` gives you levels, formatting, and multiple destinations for free
3. **Separate by responsibility** — even trivial functions get their own module if the responsibility is distinct
4. **Fail gracefully** — log and skip is a deliberate failure policy, not a missing try/except
5. **Test the pure parts** — parsers are pure functions, easy to test with known HTML strings
6. **CLI separates data from code** — URLs and output paths come from the user, not hardcoded